In [1]:
"""
HR Analytics - Synthetic Dataset Generator
Generates ~5,000 employee records and saves to Parquet.

Parquet output path:
C:\\Users\\mihir\\hr-analytics-project\\data\\hr_employees.parquet
"""

import os
from datetime import datetime, timedelta
import numpy as np
import pandas as pd
from faker import Faker

# =============================================
# CONFIGURATION
# =============================================
np.random.seed(42)
fake = Faker()

# Dataset size
N_EMPLOYEES = 5000
N_MANAGERS = 250

# Fixed paths as requested
DATA_DIR = r"C:\Users\mihir\hr-analytics-project\data"
os.makedirs(DATA_DIR, exist_ok=True)

PARQUET_PATH = os.path.join(DATA_DIR, "hr_employees.parquet")

print(f"Data Directory: {DATA_DIR}")
print(f"Parquet Path : {PARQUET_PATH}")
print(f"Generating {N_EMPLOYEES} employee records...\n")

# =============================================
# HELPER FUNCTIONS
# =============================================
def random_date(start_date, end_date):
    """Generate random date between start and end."""
    delta = end_date - start_date
    rand_days = np.random.randint(0, delta.days + 1)
    return start_date + timedelta(days=int(rand_days))

def calculate_attrition_probability(tenure_months, performance_rating, 
                                   overtime_hours_month, salary, 
                                   manager_rating, job_satisfaction):
    """
    Calculate attrition probability based on multiple factors.
    Returns a probability between 0 and 1.
    """
    base_prob = 0.08  # 8% base attrition rate
    
    # Tenure effect (newer employees more likely to leave)
    if tenure_months < 6:
        base_prob += 0.10
    elif tenure_months < 12:
        base_prob += 0.06
    elif tenure_months < 24:
        base_prob += 0.03
    elif tenure_months > 60:
        base_prob -= 0.02  # loyal employees
    
    # Performance effect
    if performance_rating <= 2:
        base_prob += 0.08
    elif performance_rating == 3:
        base_prob += 0.03
    elif performance_rating == 5:
        base_prob -= 0.02
    
    # Overtime effect
    if overtime_hours_month > 25:
        base_prob += 0.07
    elif overtime_hours_month > 15:
        base_prob += 0.04
    
    # Salary effect (relative to expectations)
    if salary < 40000:
        base_prob += 0.06
    elif salary < 55000:
        base_prob += 0.03
    elif salary > 100000:
        base_prob -= 0.02
    
    # Manager quality effect
    if manager_rating <= 2:
        base_prob += 0.06
    elif manager_rating >= 4:
        base_prob -= 0.02
    
    # Job satisfaction effect
    if job_satisfaction <= 2.0:
        base_prob += 0.10
    elif job_satisfaction <= 3.0:
        base_prob += 0.05
    elif job_satisfaction >= 4.5:
        base_prob -= 0.03
    
    return np.clip(base_prob, 0.02, 0.40)  # Keep between 2% and 40%

# =============================================
# REFERENCE DATA
# =============================================
departments = [
    "Engineering", "Sales", "Marketing", "Finance", 
    "Operations", "HR", "Customer Support", "Product", 
    "Data & Analytics", "Legal"
]

job_levels = {
    1: "Junior",
    2: "Mid-Level",
    3: "Senior",
    4: "Lead/Manager",
    5: "Director/VP"
}

education_levels = ["High School", "Associate", "Bachelors", "Masters", "PhD"]

locations = [
    "New York", "San Francisco", "Austin", "Seattle", 
    "Boston", "Chicago", "London", "Berlin", 
    "Singapore", "Bangalore", "Remote"
]

genders = ["Male", "Female", "Non-binary", "Prefer not to say"]

attrition_reasons = [
    "Better Opportunity", "Higher Compensation", "Career Change",
    "Relocation", "Work-Life Balance", "Management Issues",
    "Company Culture", "Lack of Growth", "Health Reasons",
    "Retirement", "Layoff", "Contract End"
]

# =============================================
# GENERATE MANAGERS
# =============================================
print("Generating managers...")

manager_data = []
for mgr_id in range(1, N_MANAGERS + 1):
    manager_data.append({
        "manager_id": mgr_id,
        "manager_name": fake.name(),
        "manager_department": np.random.choice(departments),
        "manager_location": np.random.choice(locations),
        "manager_rating": np.random.choice([2, 3, 4, 5], p=[0.08, 0.42, 0.35, 0.15])
    })

managers_df = pd.DataFrame(manager_data)
print(f"✓ Generated {len(managers_df)} managers")

# =============================================
# GENERATE EMPLOYEES
# =============================================
print("Generating employees...")

employees = []
start_company_date = datetime(2010, 1, 1)
end_company_date = datetime(2025, 11, 30)
reference_date = datetime(2026, 2, 1)

for emp_id in range(1, N_EMPLOYEES + 1):
    # Demographics
    age = np.random.randint(21, 65)
    gender = np.random.choice(genders, p=[0.48, 0.48, 0.02, 0.02])
    
    # Department and location
    department = np.random.choice(
        departments, 
        p=[0.24, 0.16, 0.12, 0.10, 0.12, 0.06, 0.08, 0.06, 0.04, 0.02]
    )
    location = np.random.choice(
        locations,
        p=[0.15, 0.14, 0.10, 0.10, 0.08, 0.08, 0.10, 0.08, 0.07, 0.07, 0.03]
    )
    
    # Job level
    job_level = np.random.choice(
        list(job_levels.keys()),
        p=[0.38, 0.28, 0.20, 0.10, 0.04]
    )
    
    # Education
    if job_level >= 4:
        education = np.random.choice(education_levels, p=[0.05, 0.10, 0.40, 0.35, 0.10])
    elif job_level >= 3:
        education = np.random.choice(education_levels, p=[0.10, 0.15, 0.45, 0.25, 0.05])
    else:
        education = np.random.choice(education_levels, p=[0.20, 0.20, 0.40, 0.18, 0.02])
    
    # Hire date and tenure
    hire_date = random_date(start_company_date, end_company_date)
    tenure_months = max(1, (reference_date.year - hire_date.year) * 12 + 
                       reference_date.month - hire_date.month)
    
    # Salary calculation (realistic by level and department)
    salary_base = {1: 42000, 2: 62000, 3: 85000, 4: 115000, 5: 155000}[job_level]
    
    dept_multiplier = {
        "Engineering": 1.30, "Data & Analytics": 1.28, "Product": 1.25,
        "Sales": 1.15, "Finance": 1.20, "Legal": 1.22,
        "Marketing": 1.10, "Operations": 1.05, "HR": 1.00,
        "Customer Support": 0.95
    }[department]
    
    location_bonus = {
        "San Francisco": 1.25, "New York": 1.20, "Seattle": 1.15,
        "Boston": 1.12, "Singapore": 1.10, "London": 1.08,
        "Austin": 1.05, "Chicago": 1.03, "Berlin": 1.00,
        "Bangalore": 0.70, "Remote": 0.95
    }[location]
    
    salary_noise = np.random.normal(0, 5000)
    salary = salary_base * dept_multiplier * location_bonus + salary_noise
    salary = float(np.clip(salary, 35000, 250000))
    
    # Performance rating (skewed toward 3-4)
    performance_rating = np.random.choice([1, 2, 3, 4, 5], p=[0.03, 0.12, 0.40, 0.35, 0.10])
    
    # Bonus percentage
    base_bonus = {1: 0.05, 2: 0.08, 3: 0.12, 4: 0.18, 5: 0.25}[job_level]
    perf_adjust = (performance_rating - 3) * 0.02
    bonus_pct = max(0, base_bonus + perf_adjust + np.random.normal(0, 0.01))
    
    # Training and development
    training_hours_year = np.clip(np.random.normal(30, 15), 0, 120)
    
    # Overtime
    if department in ["Engineering", "Operations", "Customer Support"]:
        overtime_mean = 12
    elif department in ["Sales", "Finance"]:
        overtime_mean = 8
    else:
        overtime_mean = 5
    
    overtime_hours_month = np.clip(np.random.normal(overtime_mean, 8), 0, 80)
    
    # Assign manager from same department (when possible)
    dept_managers = managers_df[managers_df["manager_department"] == department]
    if len(dept_managers) > 0:
        mgr = dept_managers.sample(1).iloc[0]
    else:
        mgr = managers_df.sample(1).iloc[0]
    
    manager_id = int(mgr["manager_id"])
    manager_rating = int(mgr["manager_rating"])
    
    # Satisfaction scores
    job_satisfaction = np.clip(
        np.random.normal(3.4, 0.9) + (performance_rating - 3) * 0.15,
        1, 5
    )
    
    work_life_balance = np.clip(
        np.random.normal(3.2, 0.8) - (overtime_hours_month / 40),
        1, 5
    )
    
    # Remote work
    remote_days_per_week = 0
    if location == "Remote":
        remote_days_per_week = 5
    elif np.random.rand() < 0.60:  # 60% have some remote work
        remote_days_per_week = np.random.choice([1, 2, 3], p=[0.3, 0.5, 0.2])
    
    # Attrition calculation
    attrition_prob = calculate_attrition_probability(
        tenure_months, performance_rating, overtime_hours_month,
        salary, manager_rating, job_satisfaction
    )
    
    attrition_flag = int(np.random.rand() < attrition_prob)
    attrition_date = None
    attrition_reason = None
    
    if attrition_flag == 1:
        # Attrition happened between hire and reference date
        attrition_date = random_date(hire_date + timedelta(days=30), reference_date)
        attrition_reason = np.random.choice(attrition_reasons)
    
    # Last promotion date
    last_promotion_date = None
    if tenure_months > 12:
        months_since_promo = np.random.randint(6, min(tenure_months, 48))
        last_promotion_date = reference_date - timedelta(days=months_since_promo * 30)
    
    # Sick days
    sick_days_year = int(np.clip(np.random.exponential(3), 0, 20))
    
    # Build employee record
    employees.append({
        "employee_id": emp_id,
        "full_name": fake.name(),
        "email": fake.email(),
        "age": age,
        "gender": gender,
        "department": department,
        "job_level": job_level,
        "job_title": job_levels[job_level],
        "education_level": education,
        "location": location,
        "hire_date": hire_date.date(),
        "tenure_months": tenure_months,
        "salary": round(salary, 2),
        "bonus_pct": round(bonus_pct, 3),
        "training_hours_year": round(training_hours_year, 1),
        "overtime_hours_month": round(overtime_hours_month, 1),
        "remote_days_per_week": remote_days_per_week,
        "performance_rating": performance_rating,
        "job_satisfaction": round(float(job_satisfaction), 2),
        "work_life_balance": round(float(work_life_balance), 2),
        "sick_days_year": sick_days_year,
        "last_promotion_date": last_promotion_date.date() if last_promotion_date else None,
        "manager_id": manager_id,
        "manager_rating": manager_rating,
        "attrition_flag": attrition_flag,
        "attrition_date": attrition_date.date() if attrition_date else None,
        "attrition_reason": attrition_reason
    })
    
    if emp_id % 1000 == 0:
        print(f"  Generated {emp_id}/{N_EMPLOYEES} employees...")

hr_df = pd.DataFrame(employees)
print(f"✓ Generated {len(hr_df)} employee records")

# =============================================
# DATA QUALITY CHECKS
# =============================================
print("\n" + "="*50)
print("DATA QUALITY CHECKS")
print("="*50)

print(f"Total Employees: {len(hr_df):,}")
print(f"Columns: {len(hr_df.columns)}")
print(f"\nAttrition Rate: {hr_df['attrition_flag'].mean():.1%}")
print(f"Average Salary: ${hr_df['salary'].mean():,.2f}")
print(f"Average Tenure: {hr_df['tenure_months'].mean():.1f} months")
print(f"Average Performance: {hr_df['performance_rating'].mean():.2f}/5")

print("\nEmployees by Department:")
print(hr_df['department'].value_counts().sort_index())

print("\nEmployees by Job Level:")
print(hr_df['job_level'].value_counts().sort_index())

print("\nSalary by Job Level:")
print(hr_df.groupby('job_level')['salary'].agg(['mean', 'min', 'max']).round(2))

# =============================================
# SAVE TO PARQUET
# =============================================
print("\n" + "="*50)
print("SAVING DATA")
print("="*50)

hr_df.to_parquet(PARQUET_PATH, engine="pyarrow", index=False, compression="gzip")
print(f"✓ Saved Parquet: {PARQUET_PATH}")
print(f"  File size: {os.path.getsize(PARQUET_PATH) / 1024:.1f} KB")

print("\n✅ DATA GENERATION COMPLETE!")


Data Directory: C:\Users\mihir\hr-analytics-project\data
Parquet Path : C:\Users\mihir\hr-analytics-project\data\hr_employees.parquet
Generating 5000 employee records...

Generating managers...
✓ Generated 250 managers
Generating employees...
  Generated 1000/5000 employees...
  Generated 2000/5000 employees...
  Generated 3000/5000 employees...
  Generated 4000/5000 employees...
  Generated 5000/5000 employees...
✓ Generated 5000 employee records

DATA QUALITY CHECKS
Total Employees: 5,000
Columns: 27

Attrition Rate: 11.9%
Average Salary: $84,915.86
Average Tenure: 97.3 months
Average Performance: 3.38/5

Employees by Department:
department
Customer Support     395
Data & Analytics     183
Engineering         1237
Finance              479
HR                   303
Legal                106
Marketing            608
Operations           601
Product              284
Sales                804
Name: count, dtype: int64

Employees by Job Level:
job_level
1    1958
2    1361
3    1017
4     48

In [2]:
import sqlite3
import os
import pandas as pd

# ---------------------------------
# Paths
# ---------------------------------
DATA_DIR = r"C:\Users\mihir\hr-analytics-project\data"
PARQUET_PATH = os.path.join(DATA_DIR, "hr_employees.parquet")
DB_PATH = os.path.join(DATA_DIR, "hr_analytics.db")

print("Parquet path :", PARQUET_PATH)
print("SQLite path  :", DB_PATH)

# ---------------------------------
# Load Parquet into DataFrame
# ---------------------------------
if not os.path.exists(PARQUET_PATH):
    raise FileNotFoundError(f"Parquet file not found at {PARQUET_PATH}. "
                            "Run the data generation cell first.")

hr_df = pd.read_parquet(PARQUET_PATH)
print(f"Loaded {len(hr_df):,} rows and {len(hr_df.columns)} columns from Parquet")

# ---------------------------------
# Create / Replace SQLite DB
# ---------------------------------
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)
    print("Removed existing SQLite DB")

conn = sqlite3.connect(DB_PATH)

# Write employees table
hr_df.to_sql("employees", conn, if_exists="replace", index=False)
print("Created table: employees")

# Create some helpful indexes for later API queries
cur = conn.cursor()
cur.execute("CREATE INDEX idx_employees_department ON employees(department)")
cur.execute("CREATE INDEX idx_employees_job_level ON employees(job_level)")
cur.execute("CREATE INDEX idx_employees_attrition ON employees(attrition_flag)")
cur.execute("CREATE INDEX idx_employees_location ON employees(location)")
cur.execute("CREATE INDEX idx_employees_manager ON employees(manager_id)")
conn.commit()

# Quick sanity checks
cur.execute("SELECT COUNT(*) FROM employees")
count = cur.fetchone()[0]

cur.execute("SELECT department, COUNT(*) FROM employees GROUP BY department ORDER BY department")
dept_counts = cur.fetchall()

conn.close()

print(f"\n✓ Created SQLite DB with {count:,} employees at: {DB_PATH}")
print("\nEmployees by department:")
for dept, c in dept_counts:
    print(f"  {dept:18s} -> {c:4d}")


Parquet path : C:\Users\mihir\hr-analytics-project\data\hr_employees.parquet
SQLite path  : C:\Users\mihir\hr-analytics-project\data\hr_analytics.db
Loaded 5,000 rows and 27 columns from Parquet
Created table: employees

✓ Created SQLite DB with 5,000 employees at: C:\Users\mihir\hr-analytics-project\data\hr_analytics.db

Employees by department:
  Customer Support   ->  395
  Data & Analytics   ->  183
  Engineering        -> 1237
  Finance            ->  479
  HR                 ->  303
  Legal              ->  106
  Marketing          ->  608
  Operations         ->  601
  Product            ->  284
  Sales              ->  804
